In [0]:

from pyspark.sql import functions as F

# Define the child table, its ID column, and the parent table's ID column.
relationships = [
    ("site", "StudyID", "study", "StudyID"),
    ("subject", "StudyID", "study", "StudyID"),
    ("subject", "SiteID", "site", "SiteID"),
    ("visit", "SubjectID", "subject", "SubjectID"),
    ("adverseevents", "SubjectID", "subject", "SubjectID"),
    ("labresults", "SubjectID", "subject", "SubjectID"),
    ("drugdispensation", "SubjectID", "subject", "SubjectID"),
    ("randomization", "SubjectID", "subject", "SubjectID"),
    ("shipment", "SiteID", "site", "SiteID"),
    ("investigator", "SiteID", "site", "SiteID")
]

results = []

# Check each relationship.
for child_table, child_key, parent_table, parent_key in relationships:

    child = spark.table(f"gpo_analytics.bronze.{child_table}")
    parent = spark.table(f"gpo_analytics.bronze.{parent_table}")

    # Keep distinct, non-null parent IDs for matching.
    parent_ids = (
        parent
        .filter(F.col(parent_key).isNotNull())
        .select(F.col(parent_key).alias("_parent_id"))
        .distinct()
    )

    # Find child records with a missing or unmatched ID.
    child_key_col = F.col(child_key)
    unmatched = (
        child
        .join(
            parent_ids,
            child_key_col == F.col("_parent_id"),
            "left"
        )
        .filter(
            child_key_col.isNull() |
            (F.trim(child_key_col) == "") |
            F.col("_parent_id").isNull()
        )
    )

    issue_count = unmatched.count()

    results.append((
        child_table,
        child_key,
        parent_table,
        parent_key,
        issue_count
    ))

# Display the results.
result_df = spark.createDataFrame(
    results,
    [
        "child_table",
        "child_column",
        "parent_table",
        "parent_column",
        "unmatched_or_missing_rows"
    ]
)

display(result_df)


child_table,child_column,parent_table,parent_column,unmatched_or_missing_rows
site,StudyID,study,StudyID,0
subject,StudyID,study,StudyID,0
subject,SiteID,site,SiteID,0
visit,SubjectID,subject,SubjectID,0
adverseevents,SubjectID,subject,SubjectID,0
labresults,SubjectID,subject,SubjectID,0
drugdispensation,SubjectID,subject,SubjectID,0
randomization,SubjectID,subject,SubjectID,0
shipment,SiteID,site,SiteID,0
investigator,SiteID,site,SiteID,0


In [0]:
from pyspark.sql import functions as F

study_date_check = (
    spark.table("gpo_analytics.bronze.study")
    .groupBy("Status")
    .agg(
        F.count("*").alias("total_studies"),
        F.sum(
            F.when(F.col("StudyEndDate").isNull(), 1).otherwise(0)
        ).alias("missing_end_dates")
    )
    .orderBy("Status")
)

display(study_date_check)

Status,total_studies,missing_end_dates
Active,7,0
Completed,2,0
Recruiting,9,0
Terminated,2,2


In [0]:
study_missing_end_dates = (
    spark.table("gpo_analytics.bronze.study")
    .filter(F.col("StudyEndDate").isNull())
    .select("StudyID", "StudyName", "Status", "StudyStartDate", "StudyEndDate")
)

display(study_missing_end_dates)

StudyID,StudyName,Status,StudyStartDate,StudyEndDate
ST012,Neurology Clinical Study 12,Terminated,2025-08-31,null
ST016,Immunology Clinical Study 16,Terminated,2025-11-29,null


In [0]:
subject_enrollment_check = (
    spark.table("gpo_analytics.bronze.subject")
    .groupBy("SubjectStatus")
    .agg(
        F.count("*").alias("total_subjects"),
        F.sum(
            F.when(F.col("EnrollmentDate").isNull(), 1).otherwise(0)
        ).alias("missing_enrollment_dates")
    )
    .orderBy("SubjectStatus")
)

display(subject_enrollment_check)

SubjectStatus,total_subjects,missing_enrollment_dates
Active,1160,0
Completed,484,0
Enrolled,520,0
Screen Failed,315,315
Screened,291,291
Withdrawn,230,0


In [0]:
from pyspark.sql import functions as F

shipment_delivery_check = (
    spark.table("gpo_analytics.bronze.shipment")
    .groupBy("Status")
    .agg(
        F.count("*").alias("total_shipments"),
        F.sum(
            F.when(F.col("ActualDeliveryDate").isNull(), 1).otherwise(0)
        ).alias("missing_delivery_dates")
    )
    .orderBy("Status")
)

display(shipment_delivery_check)

Status,total_shipments,missing_delivery_dates
Delayed,50,50
Delivered,381,0
In Transit,91,91
Pending,69,69
Returned,9,9


In [0]:
from pyspark.sql import functions as F

study_date_format_check = (
    spark.table("gpo_analytics.bronze.study")
    .select(
        "StudyID",
        "StudyStartDate",
        "StudyEndDate",
        F.to_date("StudyStartDate", "yyyy-MM-dd").alias("parsed_start_date"),
        F.to_date("StudyEndDate", "yyyy-MM-dd").alias("parsed_end_date")
    )
    .filter(
        (F.col("StudyStartDate").isNotNull() & F.col("parsed_start_date").isNull())
        | (F.col("StudyEndDate").isNotNull() & F.col("parsed_end_date").isNull())
    )
)

display(study_date_format_check)

StudyID,StudyStartDate,StudyEndDate,parsed_start_date,parsed_end_date


In [0]:
from pyspark.sql import functions as F

shipment_date_format_check = (
    spark.table("gpo_analytics.bronze.shipment")
    .select(
        "ShipmentID",
        "ExpectedDeliveryDate",
        "ActualDeliveryDate",
        F.to_date("ExpectedDeliveryDate", "yyyy-MM-dd").alias("parsed_expected_date"),
        F.to_date("ActualDeliveryDate", "yyyy-MM-dd").alias("parsed_actual_date")
    )
    .filter(
        (
            F.col("ExpectedDeliveryDate").isNotNull()
            & F.col("parsed_expected_date").isNull()
        )
        |
        (
            F.col("ActualDeliveryDate").isNotNull()
            & F.col("parsed_actual_date").isNull()
        )
    )
)

display(shipment_date_format_check)

ShipmentID,ExpectedDeliveryDate,ActualDeliveryDate,parsed_expected_date,parsed_actual_date


In [0]:
from pyspark.sql import functions as F

subject_date_format_check = (
    spark.table("gpo_analytics.bronze.subject")
    .select(
        "SubjectID",
        "ScreenDate",
        "EnrollmentDate",
        F.to_date("ScreenDate", "yyyy-MM-dd").alias("parsed_screen_date"),
        F.to_date("EnrollmentDate", "yyyy-MM-dd").alias("parsed_enrollment_date")
    )
    .filter(
        (
            F.col("ScreenDate").isNotNull()
            & F.col("parsed_screen_date").isNull()
        )
        |
        (
            F.col("EnrollmentDate").isNotNull()
            & F.col("parsed_enrollment_date").isNull()
        )
    )
)

display(subject_date_format_check)

SubjectID,ScreenDate,EnrollmentDate,parsed_screen_date,parsed_enrollment_date


In [0]:
from pyspark.sql import functions as F

subject_age_check = (
    spark.table("gpo_analytics.bronze.subject")
    .select(
        "SubjectID",
        "Age",
        F.trim(F.col("Age")).alias("trimmed_age"),
        F.col("Age").cast("int").alias("parsed_age")
    )
    .filter(
        F.col("Age").isNotNull()
        & (F.trim(F.col("Age")) != "")
        & F.col("parsed_age").isNull()
    )
)

display(subject_age_check)

SubjectID,Age,trimmed_age,parsed_age


In [0]:
from pyspark.sql import functions as F

subject_age_range_check = (
    spark.table("gpo_analytics.bronze.subject")
    .select(F.col("Age").cast("int").alias("Age"))
    .agg(
        F.min("Age").alias("minimum_age"),
        F.max("Age").alias("maximum_age"),
        F.sum(
            F.when((F.col("Age") < 18) | (F.col("Age") > 100), 1)
            .otherwise(0)
        ).alias("ages_outside_18_to_100")
    )
)

display(subject_age_range_check)

minimum_age,maximum_age,ages_outside_18_to_100
18,85,0


In [0]:
from pyspark.sql import functions as F

study_enrollment_check = (
    spark.table("gpo_analytics.bronze.study")
    .select(
        "StudyID",
        "PlannedEnrollment",
        F.col("PlannedEnrollment").cast("int").alias("parsed_enrollment")
    )
    .filter(
        F.col("PlannedEnrollment").isNotNull()
        & (F.trim(F.col("PlannedEnrollment")) != "")
        & (
            F.col("parsed_enrollment").isNull()
            | (F.col("parsed_enrollment") < 0)
        )
    )
)

display(study_enrollment_check)

StudyID,PlannedEnrollment,parsed_enrollment


In [0]:
from pyspark.sql import functions as F

lab_result_check = (
    spark.table("gpo_analytics.bronze.labresults")
    .select(
        "LabID",
        "Result",
        F.trim(F.col("Result")).alias("trimmed_result"),
        F.col("Result").cast("double").alias("parsed_result")
    )
    .filter(
        F.col("Result").isNotNull()
        & (F.trim(F.col("Result")) != "")
        & F.col("parsed_result").isNull()
    )
)

display(lab_result_check)

LabID,Result,trimmed_result,parsed_result


In [0]:
from pyspark.sql import functions as F

dose_check = (
    spark.table("gpo_analytics.bronze.drugdispensation")
    .select(
        "DispenseID",
        "Dose",
        F.expr("try_cast(Dose AS DOUBLE)").alias("parsed_dose")
    )
    .filter(
        F.col("Dose").isNotNull()
        & (F.trim(F.col("Dose")) != "")
        & (
            F.col("parsed_dose").isNull()
            | (F.col("parsed_dose") < 0)
        )
    )
)

display(dose_check)

DispenseID,Dose,parsed_dose
D000001,25 mg,null
D000002,100 mg,null
D000003,1 Tablet,null
D000004,50 mg,null
D000005,1 Tablet,null
D000006,100 mg,null
D000007,1 Tablet,null
D000008,50 mg,null
D000009,25 mg,null
D000010,100 mg,null


In [0]:
# Read the Bronze study table.
study_bronze = spark.table("gpo_analytics.bronze.study")

# Print the exact column names available in the table.
print(study_bronze.columns)

['StudyID', 'ProtocolID', 'StudyName', 'Phase', 'TherapeuticArea', 'Status', 'PlannedEnrollment', 'StudyStartDate', 'StudyEndDate', '_source_file', '_ingested_at']


In [0]:
from pyspark.sql import functions as F

shipment_business_rule_check = (
    spark.table("gpo_analytics.bronze.shipment")
    .filter(
        (
            (F.col("Status") == "Delivered")
            & F.col("ActualDeliveryDate").isNull()
        )
        |
        (
            (F.col("Status") != "Delivered")
            & F.col("ActualDeliveryDate").isNotNull()
        )
    )
    .select(
        "ShipmentID",
        "Status",
        "ExpectedDeliveryDate",
        "ActualDeliveryDate"
    )
)

display(shipment_business_rule_check)

ShipmentID,Status,ExpectedDeliveryDate,ActualDeliveryDate
